In [1]:
################################################################################
# Forest Disturbance Stack
# Validation of extreme disturbance overlap definitions
# 2000–2020
################################################################################

from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import rasterio


# =============================================================================
# 1. DIRECTORIES AND SETTINGS
# =============================================================================

ROOT = Path(
    "/home/jovyan/data-store/forest-disturbance-stack-v3"
)

ANNUAL_DIR = ROOT / "data/derived/annual_stacks/binary"

FOREST_PATH = (
    ROOT / "data/derived/resampled/"
    "forest_mask_30m_resampled.tif"
)

OUT_DIR = (
    ROOT / "data/derived/disturbance_sequence/extreme_sequences"
)

OUT_DIR.mkdir(parents=True, exist_ok=True)

YEARS = range(2000, 2021)

DISTURBANCES = ("WF", "BT", "HD")

PAIRS = [
    ("WF", "BT"),
    ("WF", "HD"),
    ("BT", "HD")
]

BANDS = {
    "WF": 1,
    "BT": 2,
    "HD": 3
}

PIXEL_AREA_KM2 = 0.0009

MISSING_YEAR = 32767


# =============================================================================
# 2. INITIALIZE COUNTERS
# =============================================================================

counts = Counter()

forest_total = 0


# =============================================================================
# 3. PROCESS ANNUAL RASTERS
# =============================================================================

from contextlib import ExitStack

with ExitStack() as stack:

    forest_src = stack.enter_context(
        rasterio.open(FOREST_PATH)
    )

    annual = {
        year: stack.enter_context(
            rasterio.open(
                ANNUAL_DIR /
                f"annual_stack_extreme_{year}.tif"
            )
        )
        for year in YEARS
    }

    # Confirm alignment
    for src in annual.values():

        if not (
            src.width == forest_src.width
            and src.height == forest_src.height
            and src.transform == forest_src.transform
            and src.crs == forest_src.crs
        ):
            raise ValueError(
                f"Raster alignment mismatch: {src.name}"
            )

    # -------------------------------------------------------------------------
    # Process each block
    # -------------------------------------------------------------------------

    for block_number, (_, window) in enumerate(
        forest_src.block_windows(1),
        start=1
    ):

        if block_number % 100 == 0:

            print(
                f"Processed {block_number:,} blocks",
                flush=True
            )

        forest = (
            forest_src.read(1, window=window) == 1
        )

        if not forest.any():
            continue

        shape = forest.shape

        # Track whether each disturbance occurred in any year
        ever = {
            d: np.zeros(shape, dtype=bool)
            for d in DISTURBANCES
        }

        # Track first observed year
        first_year = {
            d: np.full(
                shape,
                MISSING_YEAR,
                dtype=np.int16
            )
            for d in DISTURBANCES
        }

        # Track whether each pair occurred together in any year
        same_year = {
            pair: np.zeros(shape, dtype=bool)
            for pair in PAIRS
        }

        # ---------------------------------------------------------------------
        # Read each annual raster
        # ---------------------------------------------------------------------

        for year in YEARS:

            bands = annual[year].read(
                [1, 2, 3],
                window=window
            )

            current = {
                d: (bands[BANDS[d] - 1] == 1) & forest
                for d in DISTURBANCES
            }

            # Ever disturbed and first observed year
            for d in DISTURBANCES:

                ever[d] |= current[d]

                new = (
                    current[d]
                    & (first_year[d] == MISSING_YEAR)
                )

                first_year[d][new] = year

            # Same-year co-occurrence
            for pair in PAIRS:

                a, b = pair

                same_year[pair] |= (
                    current[a] & current[b]
                )

        # ---------------------------------------------------------------------
        # Summarize each pair
        # ---------------------------------------------------------------------

        forest_total += int(
            np.count_nonzero(forest)
        )

        for pair in PAIRS:

            a, b = pair

            spatial_overlap = (
                ever[a] & ever[b]
            )

            simultaneous_first = (
                (first_year[a] == first_year[b])
                & ever[a]
                & ever[b]
            )

            counts[(pair, "spatial_overlap")] += int(
                np.count_nonzero(spatial_overlap)
            )

            counts[(pair, "same_year_overlap")] += int(
                np.count_nonzero(same_year[pair])
            )

            counts[(pair, "simultaneous_first")] += int(
                np.count_nonzero(simultaneous_first)
            )

        # ---------------------------------------------------------------------
        # Three-disturbance overlap
        # ---------------------------------------------------------------------

        all_three = (
            ever["WF"]
            & ever["BT"]
            & ever["HD"]
        )

        counts[("all_three", "spatial_overlap")] += int(
            np.count_nonzero(all_three)
        )


# =============================================================================
# 4. VALIDATE RESULTS
# =============================================================================

for pair in PAIRS:

    n_spatial = counts[(pair, "spatial_overlap")]
    n_same = counts[(pair, "same_year_overlap")]
    n_first = counts[(pair, "simultaneous_first")]

    assert n_first <= n_same <= n_spatial, (
        f"Overlap relationship failed for {pair}"
    )

print("\nValidation passed: first-year <= same-year <= spatial")


# =============================================================================
# 5. CREATE SUMMARY TABLE
# =============================================================================

rows = []

for pair in PAIRS:

    a, b = pair

    row = {
        "disturbance_pair": f"{a} + {b}"
    }

    for metric in [
        "spatial_overlap",
        "same_year_overlap",
        "simultaneous_first"
    ]:

        n = counts[(pair, metric)]

        row[f"{metric}_pixels"] = n
        row[f"{metric}_km2"] = n * PIXEL_AREA_KM2
        row[f"{metric}_percent_forest"] = (
            100 * n / forest_total
        )

    # Proportion of spatially overlapping pixels
    # that experienced a same-year overlap
    spatial_n = counts[(pair, "spatial_overlap")]

    row["percent_spatial_overlap_same_year"] = (
        100 * counts[(pair, "same_year_overlap")] / spatial_n
        if spatial_n else np.nan
    )

    rows.append(row)

df_overlap = pd.DataFrame(rows)

output_path = (
    OUT_DIR / "extreme_overlap_validation.csv"
)

df_overlap.to_csv(
    output_path,
    index=False
)


# =============================================================================
# 6. PRINT RESULTS
# =============================================================================

print("\n" + "=" * 65)
print("EXTREME DISTURBANCE OVERLAP VALIDATION")
print("=" * 65)

print(
    f"Total forest area: "
    f"{forest_total * PIXEL_AREA_KM2:,.1f} km²"
)

print("\nOverlap summary (km²):")

print(
    df_overlap[
        [
            "disturbance_pair",
            "spatial_overlap_km2",
            "same_year_overlap_km2",
            "simultaneous_first_km2",
            "percent_spatial_overlap_same_year"
        ]
    ].round(2).to_string(index=False)
)

all_three_km2 = (
    counts[("all_three", "spatial_overlap")]
    * PIXEL_AREA_KM2
)

print(
    f"\nAll three disturbance types (spatial): "
    f"{all_three_km2:,.1f} km²"
)

print(
    "\nPreviously calculated all-three area: "
    "3,031.7 km²"
)

print("\nSaved:", output_path)

Processed 100 blocks
Processed 200 blocks
Processed 300 blocks
Processed 400 blocks
Processed 500 blocks
Processed 600 blocks
Processed 700 blocks
Processed 800 blocks
Processed 900 blocks
Processed 1,000 blocks
Processed 1,100 blocks
Processed 1,200 blocks
Processed 1,300 blocks
Processed 1,400 blocks
Processed 1,500 blocks
Processed 1,600 blocks
Processed 1,700 blocks
Processed 1,800 blocks
Processed 1,900 blocks
Processed 2,000 blocks
Processed 2,100 blocks
Processed 2,200 blocks
Processed 2,300 blocks
Processed 2,400 blocks
Processed 2,500 blocks
Processed 2,600 blocks
Processed 2,700 blocks
Processed 2,800 blocks
Processed 2,900 blocks
Processed 3,000 blocks
Processed 3,100 blocks
Processed 3,200 blocks
Processed 3,300 blocks
Processed 3,400 blocks
Processed 3,500 blocks
Processed 3,600 blocks
Processed 3,700 blocks
Processed 3,800 blocks
Processed 3,900 blocks
Processed 4,000 blocks
Processed 4,100 blocks
Processed 4,200 blocks
Processed 4,300 blocks
Processed 4,400 blocks
Proces